In [26]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('../data/original/german_credit_data.csv')

# Encode Target (1 = Bad Risk/Default, 0 = Good Risk/Repaid)
# We make 'bad' the positive class (1) because predicting defaults is our main goal.
df['target'] = df['class'].map({'bad': 1, 'good': 0})

# Dropping the old 'class' column to avoid confusion
df = df.drop('class', axis=1)

print(f"Dataset loaded. Shape: {df.shape}")
print("\nTarget distribution:")
print(df['target'].value_counts())

Dataset loaded. Shape: (1000, 21)

Target distribution:
target
0    700
1    300
Name: count, dtype: int64


In [27]:
# These failed our Mann-Whitney U and Chi-Square tests
cols_to_drop =['residence_since', 'num_dependents', 'existing_credits', 'own_telephone']
df = df.drop(columns=cols_to_drop)
print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")

Dropped 4 columns: ['residence_since', 'num_dependents', 'existing_credits', 'own_telephone']


In [28]:
# Log Transform for credit_amount (Fixing the right skew & 7.2% outliers)
df['credit_amount_log'] = np.log1p(df['credit_amount'])

In [29]:
# This solves the 0.62 multicollinearity between amount and duration
df['monthly_burden'] = df['credit_amount'] / df['duration']

In [30]:
# This captures the massive 77% default rate combo we found in EDA
df['checking_x_history'] = df['checking_status'].astype(str) + "_" + df['credit_history'].astype(str)

In [31]:
# We assign higher  scores to riskier categories
checking_risk_map = {'<0': 3, '0<=X<200': 2, 'no checking': 0, '>=200': 1}
savings_risk_map = {'<100': 3, '100<=X<500': 2, '500<=X<1000': 1, '>=1000': 0, 'no known savings': 0}

In [32]:
df['checking_risk'] = df['checking_status'].map(checking_risk_map)
df['savings_risk'] = df['savings_status'].map(savings_risk_map)
df['financial_stress_score'] = df['checking_risk'] + df['savings_risk']

In [33]:
# Regroup Job Categories (Stable vs Unstable)
# 'job' wasn't statistically significant because 4 categories diluted the signal. 
stable_jobs = ['skilled', 'high qualif/self emp/mgmt']
df['job_stable'] = df['job'].apply(lambda x: 1 if x in stable_jobs else 0)

In [34]:
# Clean up temporary/original columns we no longer need
df = df.drop(columns=['job', 'checking_risk', 'savings_risk'])



In [35]:
print(f"Feature Engineering complete! New shape: {df.shape}")
display(df.head(3))

Feature Engineering complete! New shape: (1000, 21)


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,age,other_payment_plans,housing,foreign_worker,target,credit_amount_log,monthly_burden,checking_x_history,financial_stress_score,job_stable
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,67,none,own,yes,0,7.064759,194.833333,<0_critical/other existing credit,3,1
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,22,none,own,yes,1,8.691483,123.979167,0<=X<200_existing paid,5,1
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,49,none,own,yes,0,7.648263,174.666667,no checking_critical/other existing credit,3,0


In [36]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   checking_status         1000 non-null   str    
 1   duration                1000 non-null   int64  
 2   credit_history          1000 non-null   str    
 3   purpose                 1000 non-null   str    
 4   credit_amount           1000 non-null   int64  
 5   savings_status          1000 non-null   str    
 6   employment              1000 non-null   str    
 7   installment_commitment  1000 non-null   int64  
 8   personal_status         1000 non-null   str    
 9   other_parties           1000 non-null   str    
 10  property_magnitude      1000 non-null   str    
 11  age                     1000 non-null   int64  
 12  other_payment_plans     1000 non-null   str    
 13  housing                 1000 non-null   str    
 14  foreign_worker          1000 non-null   str    
 15 